In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Base model
model_base = AutoModelForCausalLM.from_pretrained(
    "unsloth/gemma-3-1b-it",
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer_base = AutoTokenizer.from_pretrained("unsloth/gemma-3-1b-it")

# Fine-tuned model
model_ft = AutoModelForCausalLM.from_pretrained(
    "th-martinod/gemma-3-grpo",
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer_ft = AutoTokenizer.from_pretrained("th-martinod/gemma-3-grpo")

print("Models loaded!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Models loaded!


In [2]:
from datasets import load_dataset

# 30 ejemplos para evaluación rápida
ds_test = load_dataset("race", "all", split="train[10000:10030]")

print("Dataset loaded with", len(ds_test), "examples.")


README.md: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/37.4M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4934 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/87866 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4887 [00:00<?, ? examples/s]

Dataset loaded with 30 examples.


In [3]:
def build_prompt(article, question, options):
    letters = ["A","B","C","D"]
    prompt = f"Passage: {article}\n\nQuestion: {question}\n\nOptions:\n"
    for l, op in zip(letters, options):
        prompt += f"{l}. {op}\n"
    prompt += "\nThink step-by-step and answer with only the letter."
    return prompt


def mc_reward(text, gold):
    text = text.strip().upper()
    for letter in ["A","B","C","D"]:
        if letter in text[-5:]:
            pred = letter
            break
    else:
        pred = None
    return 1 if pred == gold else 0


In [4]:
def evaluate_model(model, tokenizer, dataset):
    correct = 0
    total = len(dataset)

    for i in range(total):
        ex = dataset[i]
        prompt = build_prompt(ex["article"], ex["question"], ex["options"])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
            )

        out_text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        reward = mc_reward(out_text, ex["answer"])
        correct += reward

        # imprimir progreso
        print(f"[{i+1}/{total}] reward={reward}")

    return correct / total


In [ ]:
print("Evaluating base model...")
acc_base = evaluate_model(model_base, tokenizer_base, ds_test)

print("\nEvaluating fine-tuned model...")
acc_ft = evaluate_model(model_ft, tokenizer_ft, ds_test)

print("\n=== RESULTS ===")
print("Base model accuracy:", acc_base)
print("Fine-tuned model accuracy:", acc_ft)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating base model...
[1/30] reward=0
[2/30] reward=0
[3/30] reward=1
[4/30] reward=0
[5/30] reward=1
[6/30] reward=0
[7/30] reward=0
[8/30] reward=1
[9/30] reward=0
[10/30] reward=0
[11/30] reward=0
[12/30] reward=0
[13/30] reward=1
[14/30] reward=0
[15/30] reward=1
[16/30] reward=1
[17/30] reward=1
[18/30] reward=1
[19/30] reward=1
[20/30] reward=0
[21/30] reward=0
[22/30] reward=0
[23/30] reward=0
[24/30] reward=1
[25/30] reward=1
[26/30] reward=0
[27/30] reward=1
[28/30] reward=1
[29/30] reward=1
[30/30] reward=0

Evaluating fine-tuned model...
[1/30] reward=0
[2/30] reward=0
[3/30] reward=1
[4/30] reward=0
[5/30] reward=1
[6/30] reward=0
[7/30] reward=0
[8/30] reward=1
[9/30] reward=0
[10/30] reward=0
[11/30] reward=0
[12/30] reward=0
[13/30] reward=1
[14/30] reward=0
[15/30] reward=1
[16/30] reward=1
[17/30] reward=1
[18/30] reward=1
[19/30] reward=1
[20/30] reward=0
[21/30] reward=0
[22/30] reward=0
[23/30] reward=0
[24/30] reward=1
[25/30] reward=1
[26/30] reward=0
[27/30] r

In [ ]:
examples = []

for i in range(3):
    ex = ds_test[i]
    prompt = build_prompt(ex["article"], ex["question"], ex["options"])

    # base
    base_in = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)
    base_out = model_base.generate(**base_in, max_new_tokens=20, do_sample=False)
    base_text = tokenizer_base.decode(base_out[0], skip_special_tokens=True)

    # fine-tuned
    ft_in = tokenizer_ft(prompt, return_tensors="pt").to(model_ft.device)
    ft_out = model_ft.generate(**ft_in, max_new_tokens=20, do_sample=False)
    ft_text = tokenizer_ft.decode(ft_out[0], skip_special_tokens=True)

    examples.append((ex, base_text, ft_text))

print("Saved 3 comparison examples!")


Saved 3 comparison examples!


In [ ]:
# para los 3 ejemplos en general
for idx, (ex, base, ft) in enumerate(examples):
    print(f"\n===== EXAMPLE {idx+1} =====")
    print("QUESTION:", ex["question"])
    print("OPTIONS:", ex["options"])
    print("Correct:", ex["answer"])

    print("\n--- Base model ---")
    print(base)

    print("\n--- Fine-tuned model ---")
    print(ft)
    print("\n")


In [6]:
# se me reinició el pc entonces tengo que correr de nuevo solo 3 ejemplos:
# Volver a cargar 3 ejemplos para comparación sin recalcular toda la accuracy

examples = []
for i in range(3):
    ex = ds_test[i]
    prompt = build_prompt(ex["article"], ex["question"], ex["options"])

    # base model output
    inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)
    with torch.no_grad():
        out_base = model_base.generate(
            **inputs_base,
            max_new_tokens=20,
            do_sample=False
        )
    text_base = tokenizer_base.decode(out_base[0], skip_special_tokens=True)

    # fine-tuned model output
    inputs_ft = tokenizer_ft(prompt, return_tensors="pt").to(model_ft.device)
    with torch.no_grad():
        out_ft = model_ft.generate(
            **inputs_ft,
            max_new_tokens=20,
            do_sample=False
        )
    text_ft = tokenizer_ft.decode(out_ft[0], skip_special_tokens=True)

    examples.append((ex, text_base, text_ft))

print("Ready! Examples regenerated.")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Ready! Examples regenerated.


In [7]:
for idx, (ex, base, ft) in enumerate(examples):
    print(f"\n===== EXAMPLE {idx+1} =====")
    print("QUESTION:", ex["question"])
    print("OPTIONS:", ex["options"])
    print("Correct:", ex["answer"])

    print("\n--- Base model ---")
    print(base)

    print("\n--- Fine-tuned model ---")
    print(ft)
    print("\n")



===== EXAMPLE 1 =====
QUESTION: The evidence found in the remains dating back to prehistoric Cyprus means  _  .
OPTIONS: ['human made cats pets as early as 9,500 years ago', 'human interacted with cats very early', "cats didn't appear until prehistoric Cyprus", 'when cats became domesticated']
Correct: B

--- Base model ---
Passage: Although cats may be one of the most popular pets today, little is known about how and when humans and cats set up their close relationship.
The earliest evidence for human-cat interaction dates back to prehistoric Cyprus , where the remains of a wild cat and a human -- dated 9,500 years old -- were found buried together.
A new study in the Proceedings of the National Academy of Sciences has confirmed the first direct evidence of a human-domestic cat relationship among Chinese farmers 5,300 years ago. Researchers studied the bones of cats, dogs, deer and other animals unearthed in an excavation   near a village in Central China. By using some ways, scienti